# Mem0 Code Example

[Mem0](https://mem0.ai) is a memory layer for AI: it extracts and stores memories from conversations, then lets you search/retrieve them by user or query.

**Setup:** `pip install mem0ai` and set `OPENAI_API_KEY`. For **graph memory** (Neo4j): `pip install "mem0ai[graph]"` and set `NEO4J_URL`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.

In [1]:
# Install (uncomment if needed)
%pip install mem0ai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from mem0 import Memory

# Uses OPENAI_API_KEY by default; or pass config to Memory()
# m = Memory.from_config({"llm": {"provider": "openai", "config": {"model": "gpt-4o-mini"}}})
m = Memory()

In [3]:
m.get_all(user_id="alex")

{'results': []}

## 1. Add memories from a conversation

Mem0 infers structured memories from message lists. Pass `user_id` (and optional `metadata`) to scope memories.

In [42]:
messages = [
    {"role": "user", "content": "Hi, I'm Alex. I love basketball and sci-fi movies."},
    {"role": "assistant", "content": "Nice to meet you, Alex! I'll remember that."},
    {"role": "user", "content": "I'm not into thrillers but I really like space operas."},
]

result = m.add(messages, user_id="alex", metadata={"source": "demo"})
print("Added memories:", result)

Added memories: {'results': [{'id': '15504786-3074-442f-8a94-d953d8c9861e', 'memory': 'Name is Alex', 'event': 'ADD'}, {'id': '272ce136-9611-4c88-ae0c-a59845a05576', 'memory': 'Likes basketball', 'event': 'ADD'}, {'id': '2e090ce5-79ab-4228-a996-76076f02ec52', 'memory': 'Likes sci-fi movies', 'event': 'ADD'}, {'id': 'ac8fab61-783e-411b-ba38-8dca0c7a20f6', 'memory': 'Not into thrillers', 'event': 'ADD'}, {'id': '4e3d18d7-dd5d-45d7-a45c-50a835fc5f32', 'memory': 'Likes space operas', 'event': 'ADD'}]}


## 2. Search memories by query

Retrieve relevant memories for a natural-language question (and optional filters).

In [4]:
results = m.search("What does Alex like?", user_id="alex")
for r in results.get("results", []):
    print(f"  - {r.get('memory')} (score: {r.get('score', 0):.2f})")

  - Name is Alex (score: 0.60)
  - Likes basketball (score: 0.30)
  - Likes sci-fi movies (score: 0.23)
  - Likes space operas (score: 0.21)
  - Not into thrillers (score: 0.19)


## 3. Get all memories for a user

List every stored memory for a user (optionally with filters).

In [5]:
all_memories = m.get_all(user_id="alex")
for mem in all_memories.get("results", []):
    print(f"  - {mem.get('memory')}")

  - Name is Alex
  - Likes basketball
  - Not into thrillers
  - Likes sci-fi movies
  - Likes space operas


## 5. Graph Memory (Neo4j)

[Graph Memory](https://docs.mem0.ai/open-source/features/graph-memory) adds a graph store alongside vector search: Mem0 extracts **entities and relationships** (people, places, events) and stores them in a Bolt-compatible graph (e.g. Neo4j). Search results can include related context in the `relations` array.

**Prerequisites:** Neo4j Aura (free tier) or self-hosted Neo4j. Set env vars: `NEO4J_URL`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.

In [6]:
# Install graph extra for Neo4j (and other graph backends)
%pip install "mem0ai[graph]"
%pip install python-dotenv

/opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1233) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from mem0 import Memory
from dotenv import load_dotenv
load_dotenv()

# Configure Mem0 with Neo4j as graph store (vectors still go to default vector DB).
# Export before running:
#   export NEO4J_URL="neo4j+s://<your-instance>.databases.neo4j.io"
#   export NEO4J_USERNAME="neo4j"
#   export NEO4J_PASSWORD="your-password"
config = {
    "graph_store": {
        "provider": "neo4j",
        "config": {
            "url": os.environ.get("NEO4J_URL", "neo4j+s://localhost:7687"),
            "username": os.environ.get("NEO4J_USERNAME", "neo4j"),
            "password": os.environ.get("NEO4J_PASSWORD", ""),
            "database": "neo4j",
        }
    }
}

# Skip graph if Neo4j env vars are not set (e.g. in CI or first run)
if os.environ.get("NEO4J_URL") and os.environ.get("NEO4J_PASSWORD"):
    memory_graph = Memory.from_config(config)
else:
    memory_graph = None
    print("Skipping graph: set NEO4J_URL and NEO4J_PASSWORD to use Neo4j graph storage.")

/Users/viktorzozula/MemoriesTesting/PersonaMem0/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Add a conversation that mentions people, place, and event — Mem0 extracts entities and edges.
# If you see KeyError: 'entity_type', the extraction LLM returned a malformed entity; use
# enable_graph=False for this add to skip graph writes (vector memory still works).
if memory_graph is not None:
    conversation = [
        {"role": "user", "content": "Alice met Bob at GraphConf 2025 in San Francisco."},
        {"role": "assistant", "content": "Great! Logging that connection."},
    ]
    try:
        memory_graph.add(conversation, user_id="demo-user")
        print("Added. In Neo4j Browser you can run: MATCH (p:Person)-[r]->(q) RETURN p,r,q LIMIT 5;")
    except KeyError as e:
        if "entity_type" in str(e):
            memory_graph.add(conversation, user_id="demo-user", enable_graph=False)
            print("Added (vector only). Graph add failed due to extraction format; memory still searchable.")
        else:
            raise

Added. In Neo4j Browser you can run: MATCH (p:Person)-[r]->(q) RETURN p,r,q LIMIT 5;


In [9]:
# Search with graph context; rerank=True often improves relevance.
if memory_graph is not None:
    results = memory_graph.search(
        "Who did Alice meet at GraphConf?",
        user_id="demo-user",
        limit=3,
        rerank=True,
    )
    for hit in results.get("results", []):
        print(hit.get("memory"))
    # Graph Memory adds related entities in the "relations" key (when available)
    if results.get("results") and "relations" in results.get("results", [{}])[0]:
        print("Relations:", results["results"][0].get("relations"))

Alice met Bob at GraphConf 2025 in San Francisco


**Optional:** Disable graph per call with `enable_graph=False` on `add()` or `search()` to use vector-only behaviour. Use a local graph like [Kuzu](https://docs.mem0.ai/open-source/features/graph-memory) (`provider: "kuzu"`, `config: {"db": "/tmp/mem0.kuzu"}`) to avoid Neo4j.

## 6. Reranker-Enhanced Search (Sentence Transformer)

[Reranker-Enhanced Search](https://docs.mem0.ai/open-source/features/reranker-search) runs a second scoring pass after vector retrieval so the most relevant memories are returned first. The **Sentence Transformer** provider uses a local Hugging Face cross-encoder—no API key, good quality, runs on CPU or GPU.

In [10]:
# Optional: install sentence-transformers if not already present (e.g. with mem0ai)
%pip install sentence-transformers

/opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1233) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [11]:
from mem0 import Memory

# Configure Mem0 with a Sentence Transformer reranker (local, no API cost).
# Use device="cuda" if you have a GPU; "cpu" works on Mac/Linux.
config_rerank = {
    "reranker": {
        "provider": "sentence_transformer",
        "config": {
            "model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
            "device": "mps",
            "max_length": 512,
            "top_k": 10,
        }
    }
}

m_rerank = Memory.from_config(config_rerank)

No sentence-transformers model found with name cross-encoder/ms-marco-MiniLM-L-6-v2. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1919.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# Add the same demo user so we have something to search (or skip if you already ran section 1).
messages = [
    {"role": "user", "content": "Hi, I'm Alex. I love basketball and sci-fi movies."},
    {"role": "assistant", "content": "Nice to meet you, Alex!"},
    {"role": "user", "content": "I'm not into thrillers but I really like space operas."},
]
m_rerank.add(messages, user_id="alex", metadata={"source": "demo"})
print("Memories added for alex.")

Memories added for alex.


In [13]:
# Search with rerank=True — results are reordered by the cross-encoder score.
results = m_rerank.search(
    "What does Alex like?",
    user_id="alex",
    limit=5,
    rerank=True,
)

print("Reranked results (score = reranker score):")
for r in results.get("results", []):
    print(f"  {r.get('score', 0):.4f}  {r.get('memory')}")

Reranked results (score = reranker score):
  0.6027  Name is Alex
  0.3370  Loves basketball
  0.2326  Likes sci-fi movies
  0.2052  Likes space operas
  0.1901  Not into thrillers


**Tip:** You can toggle reranking per request: `m_rerank.search(..., rerank=True)` vs `rerank=False`. Use a try/except to fall back to vector-only search if the reranker fails (e.g. out of memory). See the [reranker docs](https://docs.mem0.ai/open-source/features/reranker-search) for Cohere, Hugging Face, and LLM-based options.

## 7. Full Mem0 Stack: OpenAI LLM + Embedder, Neo4j Graph, Sentence Transformer Reranker

This chapter shows a **single `Memory` instance** configured with:

- **LLM**: OpenAI (for extraction / reasoning)
- **Embedder**: OpenAI `text-embedding-3-small`
- **Graph store**: Neo4j via Graph Memory ([docs](https://docs.mem0.ai/open-source/features/graph-memory))
- **Reranker**: local Sentence Transformer cross-encoder ([docs](https://docs.mem0.ai/open-source/features/reranker-search))

You only need to set environment variables:

- `OPENAI_API_KEY`
- `NEO4J_URL`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (for graph)

Reranker runs locally (CPU/MPS/GPU), so it does not need an extra API key.

In [5]:
import os
from mem0 import Memory

# Assumes you already ran %pip install mem0ai "mem0ai[graph]" sentence-transformers
# and set OPENAI_API_KEY and Neo4j env vars.
# On Apple Silicon you can use device="mps" instead of "cpu" for the reranker.

full_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4.1-mini", 
            # OPENAI_API_KEY is read from the environment
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            # Required so Mem0 can size the default vector store
            "embedding_dims": 1536,
        },
    },
    "graph_store": {
        "provider": "neo4j",
        "config": {
            "url": os.environ.get("NEO4J_URL", "neo4j+s://localhost:7687"),
            "username": os.environ.get("NEO4J_USERNAME", "neo4j"),
            "password": os.environ.get("NEO4J_PASSWORD", ""),
            "database": "neo4j",
        },
    },
    # Explicit vector store config; Mem0 will use Qdrant with this dimension.
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "embedding_model_dims": 1536,
        },
    },
    "reranker": {
        "provider": "sentence_transformer",
        "config": {
            "model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
            "device": "mps" if os.environ.get("MEM0_RERANK_DEVICE", "mps") == "mps" else "cpu",
            "max_length": 512,
            "top_k": 10,
        },
    },
}

# Only enable graph if Neo4j creds are present; everything else still works without it.
if os.environ.get("NEO4J_URL") and os.environ.get("NEO4J_PASSWORD"):
    mem = Memory.from_config(full_config)
else:
    cfg_no_graph = full_config.copy()
    cfg_no_graph.pop("graph_store", None)
    mem = Memory.from_config(cfg_no_graph)
    print("Neo4j env vars not set — running without graph_store, but reranker + OpenAI still work.")

No sentence-transformers model found with name cross-encoder/ms-marco-MiniLM-L-6-v2. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
mem.add(messages, user_id="alex", metadata={"source": "demo"})

{'results': [{'id': '1edf1d14-a0cf-4a18-a333-9884a27f045b',
   'memory': 'Name is Alex',
   'event': 'ADD'},
  {'id': '8636c76a-c18f-4817-94db-bd89150ab92c',
   'memory': 'Loves basketball',
   'event': 'ADD'},
  {'id': '2ad92d9d-cad1-49f8-b489-d2c4b70578b4',
   'memory': 'Likes sci-fi movies',
   'event': 'ADD'},
  {'id': '7bd6ee6e-039a-4949-9714-afed6bdd8917',
   'memory': 'Not into thrillers',
   'event': 'ADD'},
  {'id': '3a485191-ed6f-4601-a05e-932f865fa4a5',
   'memory': 'Really likes space operas',
   'event': 'ADD'}],
 'relations': {'deleted_entities': [],
  'added_entities': [[{'source': 'alex',
     'relationship': 'likes',
     'target': 'basketball'}],
   [{'source': 'alex', 'relationship': 'likes', 'target': 'sci-fi_movies'}],
   [{'source': 'alex', 'relationship': 'dislikes', 'target': 'thrillers'}],
   [{'source': 'alex', 'relationship': 'likes', 'target': 'space_operas'}]]}}

You now have a **single Mem0 client** (`mem`) that:

- Uses **OpenAI** for extraction and embeddings via `OPENAI_API_KEY`.
- Optionally writes to **Neo4j graph memory** when `NEO4J_*` env vars are set.
- Uses a local **Sentence Transformer reranker** to reorder results when `rerank=True`.

This is a good starting template for production: adjust model names, Neo4j URL, and the reranker device (`cpu` / `mps` / `cuda`) to match your environment.

## 8. Benchmark: Async Mem0 + batched concurrent adds (e.g. 200 users × 200 messages)

For large benchmarks (e.g. **200 users**, each with **200 messages**), use **AsyncMemory** and run many `add()` calls **concurrently** with a **semaphore** to cap in-flight OpenAI requests and avoid rate limits (RPM/TPM). Mem0 does not expose a single `batch_add()`; batching is done by scheduling many async `add()` calls and limiting concurrency.

- Same **full stack config** as chapter 7 (OpenAI LLM + embedder, Qdrant, optional Neo4j, Sentence Transformer reranker).
- **AsyncMemory** from Mem0: `await AsyncMemory.from_config(full_config)`.
- **Concurrency**: `asyncio.gather()` with a **semaphore** (e.g. 10–20) so you don’t overwhelm the OpenAI API.
- Scale: set `NUM_USERS` and `MESSAGES_PER_USER` below; the notebook uses small defaults so it runs quickly—switch to 200/200 for a full benchmark.

In [45]:
import os
import asyncio
import time
from mem0 import AsyncMemory

# Use full_config from chapter 7 (run that cell first). Otherwise we build a minimal config here.
try:
    _ = full_config
except NameError:
    full_config = {
        "llm": {"provider": "openai", "config": {"model": "gpt-4.1-mini"}},
        "embedder": {"provider": "openai", "config": {"model": "text-embedding-3-small", "embedding_dims": 1536}},
        "vector_store": {"provider": "qdrant", "config": {"embedding_model_dims": 1536}},
        "reranker": {
            "provider": "sentence_transformer",
            "config": {"model": "cross-encoder/ms-marco-MiniLM-L-6-v2", "device": "cpu", "max_length": 512, "top_k": 10},
        },
    }
    if os.environ.get("NEO4J_URL") and os.environ.get("NEO4J_PASSWORD"):
        full_config["graph_store"] = {
            "provider": "neo4j",
            "config": {
                "url": os.environ.get("NEO4J_URL"),
                "username": os.environ.get("NEO4J_USERNAME", "neo4j"),
                "password": os.environ.get("NEO4J_PASSWORD"),
                "database": "neo4j",
            },
        }

# Benchmark shape: NUM_USERS add() calls, each with MESSAGES_PER_USER messages.
# For a full run use e.g. NUM_USERS=200, MESSAGES_PER_USER=200; keep MAX_CONCURRENT_ADDS modest to avoid rate limits.
NUM_USERS = 5
MESSAGES_PER_USER = 20
MAX_CONCURRENT_ADDS = 10

def make_fake_messages(n: int):
    return [
        {"role": "user" if i % 2 == 0 else "assistant", "content": f"Message {i}: user preference or reply."}
        for i in range(n)
    ]

async def add_one_user(amem, user_id: str, messages: list, sem: asyncio.Semaphore):
    async with sem:
        return await amem.add(messages, user_id=user_id, metadata={"benchmark": True})

async def run_benchmark():
    cfg = full_config.copy()
    if not (os.environ.get("NEO4J_URL") and os.environ.get("NEO4J_PASSWORD")):
        cfg.pop("graph_store", None)
    amem = await AsyncMemory.from_config(cfg)
    sem = asyncio.Semaphore(MAX_CONCURRENT_ADDS)
    tasks = [
        add_one_user(amem, f"bench_user_{i}", make_fake_messages(MESSAGES_PER_USER), sem)
        for i in range(NUM_USERS)
    ]
    t0 = time.perf_counter()
    results = await asyncio.gather(*tasks, return_exceptions=True)
    elapsed = time.perf_counter() - t0
    ok = sum(1 for r in results if not isinstance(r, Exception))
    errs = [r for r in results if isinstance(r, Exception)]
    print(f"Done: {ok}/{NUM_USERS} users in {elapsed:.1f}s ({NUM_USERS * MESSAGES_PER_USER} messages total).")
    if errs:
        print(f"Errors: {len(errs)} — {errs[:3]}")
    return results

In [ ]:
# Run the benchmark (uses NUM_USERS, MESSAGES_PER_USER, MAX_CONCURRENT_ADDS from above).
results = asyncio.run(run_benchmark())

**Scaling to 200 users × 200 messages:** Set `NUM_USERS = 200`, `MESSAGES_PER_USER = 200`, and tune `MAX_CONCURRENT_ADDS` (e.g. 10–20) to stay under your OpenAI RPM/TPM limits. Each `add()` triggers LLM extraction and embedding calls; the semaphore batches those calls so they run concurrently without overloading the API. See [Async Memory](https://docs.mem0.ai/open-source/features/async-memory) for more.